In [1]:
from ipywidgets import interact

import cartopy.crs as ccrs
import matplotlib.pyplot as plt

import gplately

from lib.main import *

In [2]:
# Timespan for analysis
temporal_resolution = 1
time_min = 0
time_max = 1800
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

### Method 1: Download Model Using GPlately

In [ ]:
gdownload = gplately.download.DataServer("shirmard2025", verbose=False)
rotation_model, topology_features, static_polygons = gdownload.get_plate_reconstruction_files()
plate_model = gplately.PlateReconstruction(rotation_model, topology_features, static_polygons)
coastlines, continents, COBs = gdownload.get_topology_geometries()
gplot = gplately.PlotTopologies(model, coastlines=coastlines, continents=continents, COBs=COBs)

projection = ccrs.Mollweide(central_longitude=60)

### Method 2: Download Model from Plate Model Manager

In [ ]:
from plate_model_manager import PlateModelManager
PMM = PlateModelManager()

plate_model = PMM.get_model("shirmard2025", data_dir="plate_model")
rotation_files = plate_model.get_rotation_model()
topology_files = plate_model.get_layer("Topologies")
# static_polygons = plate_model.get_layer("StaticPolygons")
coastlines = plate_model.get_layer("Coastlines")
continents = plate_model.get_layer("ContinentalPolygons")
COBs = plate_model.get_layer("COBs")

plate_reconstruction = gplately.PlateReconstruction(rotation_model=rotation_files, topology_features=topology_files)
gplot = gplately.PlotTopologies(plate_reconstruction, continents=continents, COBs=COBs, coastlines=coastlines)

projection = ccrs.Mollweide(central_longitude=60)

### Method 3: Use Local Files

In [ ]:
plate_model_dir = "plate_model"

rotation_features = [
    plate_model_dir+"/1800_1000_rotfile_20240725_run3.rot",
    plate_model_dir+"/1000_0_rotfile_20240725_run3.rot",
]

topology_features = [
    plate_model_dir+"/1800-1000_plate_boundaries.gpml",
    plate_model_dir+"/250-0_plate_boundaries.gpml",
    plate_model_dir+"/410-250_plate_boundaries.gpml",
    plate_model_dir+"/1000-410-Convergence.gpml",
    plate_model_dir+"/1000-410-Divergence.gpml",
    # plate_model_dir+"/1000-410-plate-boundaries.gpml",
    plate_model_dir+"/1000-410-Topologies.gpml",
    plate_model_dir+"/1000-410-Transforms.gpml",
    plate_model_dir+"/TopologyBuildingBlocks.gpml",
]

continent_features = [
    plate_model_dir+"/shapes_continents.gpmlz",
]


model = gplately.PlateReconstruction(rotation_model=rotation_features, topology_features=topology_features)

COBs = plate_model_dir+"/COBfile_1800_0.gpml"
continents = plate_model_dir+"/shapes_continents.gpmlz"
coastlines = plate_model_dir+"/shapes_coasts.gpmlz"
gplot = gplately.PlotTopologies(model, continents=continents, COBs=COBs, coastlines=coastlines)

projection = ccrs.Mollweide(central_longitude=60)

### Method 4: My Functions

In [3]:
plate_model_name = 'shirmard2025'
# plate_model_name = None
plate_model_dir = 'plate_model'

plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

gplot = get_plot_topologies(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
    plate_reconstruction=plate_model,
    filter_topologies=True,
)

projection = ccrs.Mollweide(central_longitude=60)

In [4]:
@interact
def show_map(time=time_steps):
    gplot.time = time
    
    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")
    
    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # gplot.plot_all_topological_sections(ax, color="orangered", alpha=0.3, zorder=4)
    gplot.plot_all_topologies(ax, color="orangered", zorder=4)
    # gplot.plot_ridges_and_transforms(ax, color="orangered", alpha=0.3, zorder=4)
    gplot.plot_trenches(ax, color="dimgray", zorder=5)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", alpha=0.3, zorder=6)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=7)

    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
                
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, …